# 02 · Supervised fine-tuning (SFT)

Same loss as pre-training. Two changes:
1. **Data** is now (prompt, ideal response) pairs, formatted with a chat template.
2. **Weights**: 0 on the prompt, 1 on the response. We only learn to *answer*, not to *ask*.

Task for the rest of the talk: GSM8K math, answered in `\boxed{}` **within 256 tokens**.
The GSM8K reference solutions are short, so imitating them teaches brevity.

In [ ]:
from common import *
from tinker_cookbook.hyperparam_utils import get_lr

train = load_gsm8k("train").shuffle(seed=0)
BATCH_SIZE, STEPS, LR = 32, 40, get_lr(MODEL)

ex = train[0]
show("question", ex["question"])
show("reference solution (reformatted)", gsm8k_solution(ex["answer"]))

In [ ]:
# ── LIVE: the only real change from 01 is the weights ─────────────────────
def sft_datum(question, answer):
    convo = gsm8k_convo(question) + [{"role": "assistant", "content": gsm8k_solution(answer)}]
    model_input, weights = renderer.build_supervised_example(
        convo, train_on_what=renderers.TrainOnWhat.LAST_ASSISTANT_MESSAGE
    )
    return make_datum(model_input.to_ints(), weights.tolist())

In [ ]:
# Look at what the mask does: green = trained on, grey = context only.
d = sft_datum(ex["question"], ex["answer"])
for tok, w in zip(d.loss_fn_inputs["target_tokens"].to_torch().tolist(), d.loss_fn_inputs["weights"].to_torch().tolist()):
    print(("\033[32m" if w > 0 else "\033[90m") + tokenizer.decode([tok]), end="\033[0m")
print()

In [ ]:
# ── LIVE: same loop as 01 ─────────────────────────────────────────────────
training_client = new_training_client()

for step in range(STEPS):
    rows = train.select(range(step * BATCH_SIZE, (step + 1) * BATCH_SIZE))
    batch = [sft_datum(q, a) for q, a in zip(rows["question"], rows["answer"])]
    fwd_bwd = training_client.forward_backward(batch, loss_fn="cross_entropy")
    optim = training_client.optim_step(adam(LR))
    print(f"step {step:2d}  nll/token = {mean_nll(fwd_bwd.result(), batch):.3f}")
    optim.result()

In [ ]:
sampling_client = training_client.save_weights_and_get_sampling_client().result()
eval_gsm8k(sampling_client, label="2. SFT")
save_checkpoint(training_client, "sft")  # RL + distillation can start from here

**Takeaway:** SFT = imitation. The model can only become as good as the
demonstrations, and it only ever sees *correct* paths, never its own mistakes.